# Day 011：`generate()` 调用链与生成准备阶段

本 Notebook 与 [Day 011 互动档案](../day-011.md) 配套。今天的目标不是把 Transformers 通用库逐行读完，而是定位真实调用路径、理解 `generate()` 在进入循环前准备了什么，并为下一天的 MiniMind 核心实现精读建立入口。

本 Notebook 默认不加载完整模型、不执行长时间生成。所有代码 Cell 都可以先独立运行；最后的可选 Cell 只做轻量函数签名观察。

## 1. 两条路径必须分开

本次真实运行使用 `--load_from ./minimind-3`，所以实际路径是：

```text
eval_llm.py
-> Qwen3ForCausalLM
-> Transformers GenerationMixin.generate()

原理精读路径：
MiniMindForCausalLM.generate()
-> MiniMindForCausalLM.forward()
```

Transformers 只看调用阶段和接口契约；MiniMind 核心实现才逐行精读。

In [ ]:
from pathlib import Path

candidates = [Path.cwd(), Path.cwd().parent.parent, Path('/home/zcf/githubs/minimind')]
repo_root = next(path for path in candidates if (path / 'minimind' / 'eval_llm.py').exists())
project = repo_root / 'minimind'
eval_path = project / 'eval_llm.py'
native_path = project / 'model' / 'model_minimind.py'
print('项目根目录：', project)
print('eval_llm.py：', eval_path.exists())
print('原生模型源码：', native_path.exists())

## 2. 从 `eval_llm.py` 找到调用点

外层调用点是：

```python
generated_ids = model.generate(
    inputs=inputs['input_ids'],
    attention_mask=inputs['attention_mask'],
    max_new_tokens=args.max_new_tokens,
    do_sample=True,
    ...
)
```

先读取周围源码，确认调用者传入了什么；不执行 `main()`。

In [ ]:
lines = eval_path.read_text(encoding='utf-8').splitlines()
for number in range(78, 92):
    print(f'{number + 1:>3}: {lines[number]}')

## 3. `generate()` 不是一次计算完整答案

如果要生成 3 个新 token，至少要完成 3 次“预测下一个 token”：

```text
当前序列 -> 一次模型计算 -> 选一个 token -> 拼回序列
                                  |
                                  +-> 未结束，继续下一轮
```

`generate()` 负责循环、拼接和停止；`forward()` 只负责其中一轮模型计算。

In [ ]:
input_ids = ['我', '正在', '学习']
new_tokens = ['大', '模型', '！']
sequence = input_ids.copy()
for step, token in enumerate(new_tokens, start=1):
    sequence.append(token)
    print(f'第 {step} 次预测后：', sequence)

## 4. 模型配置和生成配置不是一回事

```text
Qwen3Config：      hidden_size、num_hidden_layers 等模型结构
GenerationConfig： max_new_tokens、temperature、top_p、do_sample 等本次设置
```

调用时显式传入的生成参数会覆盖默认值。例如默认 temperature 为 0.85，本次传入 0.1，实际使用 0.1；这只影响本次生成，不修改模型权重。

In [ ]:
model_config = {'hidden_size': 768, 'num_hidden_layers': 8}
default_generation = {'temperature': 0.85, 'top_p': 0.95}
call_overrides = {'temperature': 0.1, 'top_p': 0.9, 'max_new_tokens': 64}
effective_generation = {**default_generation, **call_overrides}
print('模型结构配置：', model_config)
print('默认生成配置：', default_generation)
print('本次实际生成配置：', effective_generation)

## 5. 生成模式和 `num_beams`

本次关键值：

```text
do_sample=True
num_beams=1
```

因此使用普通采样。`num_beams=1` 表示只维护一条生成路径，不表示词表中只有一个候选 token。Beam Search 的多路径评分和剪枝留到真正使用 `num_beams>1` 时再学。

In [ ]:
do_sample = True
num_beams = 1
strategy = '普通采样' if do_sample and num_beams == 1 else '需要进一步判断'
print('生成策略：', strategy)
print('维护的序列路径数：', num_beams)
print('词表候选数仍由模型词表决定，而不是 num_beams')

## 6. `generate()` 中间准备阶段的调用栈

在进入 `_sample()` 前，Transformers 会依次完成这些准备：

```text
准备 GenerationConfig
-> 检查 model_kwargs
-> 检查生成模式组合
-> 准备 logits_processor / stopping_criteria 容器
-> 确认 input_ids、batch_size、device
-> 准备特殊 token 和 attention_mask
-> 准备长度边界和 cache
-> 调用 _sample()
```

当前真实输入：`input_ids.shape=[1,21]`、设备为 CPU，`max_new_tokens=64`，因此最大总长度为 85。

In [ ]:
input_shape = (1, 21)
batch_size = input_shape[0]
input_length = input_shape[1]
max_new_tokens = 64
max_length = input_length + max_new_tokens
print('batch_size：', batch_size)
print('当前输入长度：', input_length)
print('device：', 'cpu')
print('最大总长度：', max_length)

## 7. 参数检查和输入检查的职责

`_validate_model_kwargs()` 只检查传给模型的参数是否合法；未知参数会在模型计算前报错。

`_validate_generation_mode()` 检查生成模式组合是否合法，例如当前实现不允许 Beam Search 与 streamer 同时使用。

`accepts_attention_mask=True` 只表示 `forward()` 的函数签名能接收这个名字，不表示已经执行了 Attention。

In [ ]:
known_kwargs = {'attention_mask', 'input_ids', 'use_cache'}
given_kwargs = {'attention_mask'}
unknown_kwargs = given_kwargs - known_kwargs
print('本次未知参数：', unknown_kwargs)
print('参数检查：', '通过' if not unknown_kwargs else '报错')
print('accepts_attention_mask：', True)
print('这只是函数签名检查，不是模型计算')

## 8. 原生 MiniMind `generate()` 的精读入口

下一天从这个函数重新建立完整链路，而不是从孤立的中间代码开始：

```python
# model/model_minimind.py
@torch.inference_mode()
def generate(...):
    input_ids = kwargs.pop('input_ids', inputs).repeat(
        num_return_sequences, 1
    )
```

今天只定位到这里，没有把 `repeat()` 算作已经掌握。Day 12 先确认原生权重/模型路径是否可用，再从运行命令、`eval_llm.py` 分支、模型实例化、原生 `generate()`、`forward()` 一路追踪。

In [ ]:
native_lines = native_path.read_text(encoding='utf-8').splitlines()
for number in range(254, 268):
    print(f'{number + 1:>3}: {native_lines[number]}')

## Day 011 自检

- 生成 3 个 token 为什么需要 3 次预测？
- `generate()` 和 `forward()` 分别负责什么？
- `Qwen3Config` 与 `GenerationConfig` 如何区分？
- `num_beams=1` 代表什么，不代表什么？
- 为什么 `input_ids.shape=[1,21]` 时最大总长度是 85？
- 为什么今天不继续逐行阅读 Transformers 的 `_validate_model_kwargs()`？

下一学习日入口：从一条原生 MiniMind 运行命令重新建立完整调用链，再逐行阅读 `MiniMindForCausalLM.generate()`。